In [0]:
%run "/Workspace/utilities"

In [0]:
dbutils.widgets.text("load_date", "")
dbutils.widgets.dropdown("load_type", "INCREMENTAL", ["INCREMENTAL", "FULL"])

load_date = dbutils.widgets.get("load_date")
load_type = dbutils.widgets.get("load_type")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Configuration

# COMMAND ----------

# Use config from utilities
logger = DataLogger("transform_order_items")

source_path = config.get_source_path("order_items.csv")
silver_path = config.get_silver_path("order_items")

logger.info(f"Source path: {source_path}")
logger.info(f"Silver path: {silver_path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Read Source Data

# COMMAND ----------

try:
    df_source = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(source_path)
    
    logger.info(f"Read {df_source.count()} order items from source CSV")
    display(df_source.limit(5))
    
except Exception as e:
    logger.error(f"Failed to read source data", e)
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Transformation

# COMMAND ----------

from pyspark.sql.functions import *

# Step 1: Validate data quality
df_valid = df_source \
    .filter(col("quantity") > 0) \
    .filter(col("unit_price") > 0) \
    .filter(col("total_price") > 0)

# Step 2: Cast numeric columns
df_cleaned = df_valid \
    .withColumn("quantity", col("quantity").cast("integer")) \
    .withColumn("unit_price", col("unit_price").cast("decimal(10,2)")) \
    .withColumn("total_price", col("total_price").cast("decimal(10,2)"))

# Step 3: Add audit columns
df_final = df_cleaned \
    .withColumn("created_date", current_timestamp()) \
    .withColumn("modified_date", current_timestamp())

logger.info(f"Transformed {df_final.count()} valid order items")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Write to Silver Layer

# COMMAND ----------

try:
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .save(silver_path)
    
    logger.info(f"✅ Successfully wrote {df_final.count()} order items to Silver layer")
    
except Exception as e:
    logger.error(f"Failed to write to Silver layer", e)
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Quality Summary

# COMMAND ----------

print(f"\n📊 Transformation Summary:")
print(f"Source records: {df_source.count()}")
print(f"Invalid records removed: {df_source.count() - df_valid.count()}")
print(f"Final records: {df_final.count()}")

summary = df_final.agg(
    sum("quantity").alias("total_quantity"),
    sum("total_price").alias("total_revenue"),
    avg("unit_price").alias("avg_unit_price")
).collect()[0]

print(f"\nTotal Quantity: {summary['total_quantity']}")
print(f"Total Revenue: ${summary['total_revenue']:.2f}")
print(f"Avg Unit Price: ${summary['avg_unit_price']:.2f}")
